# LLM Metrics: как измеряют языковые модели

Как мы понимаем, что одна модель лучше другой. На первый взгляд вопрос кажется техническим и второстепенным, но на практике именно метрики задают направление всему развитию области: модели оптимизируют то, что мы умеем измерять

Главная сложность в том, что язык "открыт". Если у арифметической задачи есть только один правильный ответ, у запроса «перескажи этот текст» или «напиши письмо коллеге» правильных ответов бесконечно много, и они могут не совпадать ни одним словом. Поэтому значительная часть истории метрик — это попытки обойти эту проблему

Два типа метрик:
- отслеживать прогресс во время обучения: здесь нужна дешёвая, автоматическая, чувствительная метрика, которую можно считать на каждом шаге
- сравнивать готовые модели по их способностям: здесь важна не дешевизна, а то, насколько метрика отражает реальную полезность

Внутренние метрики хорошо решают первую задачу, бенчмарки и человеческая оценка — вторую.

Также никто не отменял закон Гудхарта: "Как только метрика становится целью, она перестаёт быть хорошей метрикой". В области LLM он работает особенно жёстко

## Внутренние метрики: перплексия и кросс-энтропия

Языковая модель в своей основе — это модель вероятности следующего токена. Она присваивает каждой последовательности вероятность, разложенную по цепочке условных вероятностей каждого токена при условии предыдущих. Естественная мера качества такой модели — насколько высокую вероятность она присваивает реальному тексту, которого раньше не видела. Если модель хорошо понимает язык, настоящие тексты должны быть для неё «ожидаемыми».

Формально это выражается через кросс-энтропию — среднее отрицательное лог-правдоподобие токенов на отложенной выборке:

$$H = -(1/N) \sum \log P(x_i | x_1, ..., x_{i-1})$$

Перплексия — это просто экспонента от кросс-энтропии:

$$
PPL = e^{H}
$$

Мы рассматривали перплексию в первой главе

Интуиция следующая: это «эффективное число равновероятных вариантов», между которыми модель в среднем колеблется на каждом шаге. Перплексия 1 означала бы идеальное предсказание (модель всегда уверена в правильном токене), а перплексия, равная размеру словаря, — полное незнание языка (модель угадывает наугад). Чем ниже перплексия, тем лучше. Историческими ориентирами служили, например, наборы вроде Penn Treebank и WikiText, на которых десятилетиями мерили прогресс языкового моделирования.

У перплексии есть важная техническая оговорка. Она зависит от токенизации и словаря, поэтому перплексии двух моделей с разными токенизаторами напрямую несравнимы. Чтобы обойти это, используют метрики, нормированные на символы или байты — bits-per-character и bits-per-byte, — которые не зависят от того, как именно текст разбит на токены, и позволяют честно сравнивать разные архитектуры.

Сильные стороны перплексии — дешевизна и отсутствие необходимости в разметке: её можно считать на любом корпусе и строить по ней кривые обучения. Именно поэтому она остаётся главной метрикой на этапе предобучения. Слабость в том, что перплексия измеряет правдоподобие и беглость, но не полезность. Модель может прекрасно предсказывать токены и при этом плохо следовать инструкциям, врать или быть бесполезной в диалоге. Поэтому с переходом к инструктивным и диалоговым моделям перплексия перестала быть достаточной, и центр тяжести сместился к внешним, поведенческим метрикам.

---

## Сравнение с эталоном

Первые метрики возникли в машинном переводе и реферировании. Идея простая: прогнать модель на предвариательно размеченых данных и сравнить его с одним или несколькими эталонными ответами, написанными человеком. Качество сводится к мере совпадения с эталоном<Br><br>

Метрика __BLEU__ предложенна для оценки качества перевода (Machine Translation). Она измеряет совпадение набора n-грамм из ответа модели и из эталонного перевода. Большие значения сотвествуют более точному переводу:

$$\text{BLEU} = \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

где $p_n$ — модифицированная точность для n-грамм, $w_n$ — веса (обычно $w_n = 1/N$), 

Так же метрика штрафует за слишком короткие ответы (компонент BP).

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

где BP (brevity penalty) - штраф, $c$ — длина кандидата, $r$ — длина референса:

$$BP = \begin{cases} 1 & \text{если } c > r \\ e^{(1 - r/c)} & \text{если } c \le r \end{cases}$$

Метрика __ROUGE__ возникла в задачах реферирования и наоборот, ориентирована на полноту: насколько n-граммы и общие подпоследовательности эталона покрыты в ответе модели. По всем н-граммам всех эталонных суммаризаций считается кол-во совпадений с сумаризацией, предложенной моделью

$$\text{ROUGE-N} = \frac{\sum_{S \in \text{Ref}} \sum_{\text{gram}_n \in S} \text{Count}_{\text{match}}(\text{gram}_n)}{\sum_{S \in \text{Ref}} \sum_{\text{gram}_n \in S} \text{Count}(\text{gram}_n)}$$

Метрика **ROUGE-L** - модификация ROUGE, ориентированная сразу и на Precision, и на Recall. Она сравнивает общие последовательности токенов (LCS) и возвращает среднее геометрическое точности и полноты: F-меру:
$$R_{lcs} = \frac{LCS(X, Y)}{m}, \quad P_{lcs} = \frac{LCS(X, Y)}{n}, \quad F_{lcs} = \frac{(1 + \beta^2) R_{lcs} P_{lcs}}{R_{lcs} + \beta^2 P_{lcs}}$$

где $m$ — длина референса, $n$ — длина кандидата.<br><br>

Метрика __METEOR__ учитывает синонимы и словоформы, 

$$\text{METEOR} = F_{mean} \cdot (1 - \text{Penalty})$$

$$F_{mean} = \frac{P \cdot R}{\alpha P + (1 - \alpha) R}, \quad \text{Penalty} = \gamma \left(\frac{\text{chunks}}{\text{matches}}\right)^{\theta}$$

где $P$ и $R$ — точность и полнота по униграммам, chunks — число смежных сопоставленных фрагментов<br><br>

Метрика __chrF__ также замеряет F-меру попадания модели в эталон, но работает не на словарных, а на символьных n-граммах

$$\text{chrF}\beta = (1 + \beta^2) \cdot \frac{\text{chrP} \cdot \text{chrR}}{\beta^2 \cdot \text{chrP} + \text{chrR}}$$

где chrP и chrR — точность и полнота по символьным n-граммам; $\beta$ задаёт вес полноты (chrF3 при $\beta=3$)<br><br>

В задачах с коротким фактическим ответом (например, вопросно-ответные системы на SQuAD) используют точное совпадение и токенную __F-меру__

Все эти метрики легко считаются, воспроизводимы и автоматизируемы, поэтому они десятилетиями были стандартом. Но у них есть фундаментальный изъян: они поверхностные - они меряют совпадение слов, а не смысл. Корректный перефраз, не совпадающий с эталоном лексически, получает низкую оценку, а бессмысленный текст с правильными словами — завышенную

---

## Метрики на основе эмбеддингов и моделей

Метрики стали опираться на представления, выученные самими нейросетями

__BERTScore__ сопоставляет ответ и эталон не по выбранным токенам, а по близости их контекстных эмбеддингов - обогащенных описаний, генерируемых на последнем слое Трансформера непосредственно перед выбором конкретного токена. Выходные эмбединги содержат всю богатую семантику токена, благодаря этому улавливает синонимию и перефразирование

Ещё дальше идут обучаемые метрики - BLEURT и COMET (последняя стала фактическим стандартом в машинном переводе): это отдельные модели, обученные на человеческих оценках качества, то есть метрика буквально предсказывает, как ответ оценил бы человек

Модель __BLEURT__ реализует один линейный слой поверх предобученой BERT модели, которой на вход подается пара из перевода тестируемой модели $x'$ и эталонного перевода $x$. Модель учится предсказывать качество перевода на экспертной разметке

Модель __COMET__ также обучаемая метрика, использующая источник $s$, гипотезу $h$ и референс $r$. В estimator-модели эмбеддинги кодируются, затем комбинируются:

$$\mathbf{x} = [\,h; r;\; h \odot r;\; |h - r|;\; h \odot s;\; |h - s|\,]$$

$$\text{score} = f_{\theta}(\mathbf{x})$$

где $h, r, s$ — пулинговые эмбеддинги из энкодера (например, XLM-R), $\odot$ — поэлементное произведение, $f_\theta$ — feed-forward регрессор, обученный предсказывать человеческие оценки (DA/MQM).



Кульминация этой линии — подход __LLM-as-a-judge__, который к середине 2020-х стал доминирующим способом оценки открытой генерации. Берется какая-то мощная языковая модель (из топа), получает запрос, один или несоклько сгенерированных ответов для оценку + инструкцию по оцениванию, например, можно попросить выставить каждому решению свой score бал или просто выбрать лучший вариант.

У такого подхода к оценке появляется свойство масштабирумости - можно разметить миллионы генераций, не тратя много денег и времени. Качество при этом будет коррелировать с человеческой разметкой.

У модели оценщика однако тоже могут быть свои систематические искажения, о которых важно знать: 
- предпочтение собственного стиля и собственных ответов
- смещение в сторону более длинных и уверенно звучащих ответов
- чувствительность к порядку предъявления вариантов

Поэтому LLM-as_Judge важно калибровать. Как минимум перемешивать порядок вариантов и перепроверять часть примеров оценщиками

## Точные метрики

Параллельно с метриками генерации всегда существовали и простые метрики для задач с проверяемым ответом, и именно они лежат в основе большинства современных бенчмарков. 

Для задач классификации это хорошо изветсные в машинном обучении метрики типа точности (__accuracy__), а также __precision__, __recall__ и __F-мера__

Для задач генерации кода ключевой метрикой стала __pass@k__: сгенерированный код запускают на модульных тестах и проверяют функциональную корректность, а pass@k оценивает вероятность того, что среди k попыток хотя бы одна пройдёт все тесты

Для задач математики (`2 x 2 = ?`) используют классификационные метрики - проверяется точное совпадение финального ответа. Если попали в точный ответ - задача решена, не попали - не решена

Для сравнительных оценок ("какой из двух ответов лучше A или B?") — долю побед (__win rate__) одной модели над другой

Когда предполагается, что у задачи есть объективно проверяемый ответ, такие метрики наиболее надёжные. Сейчас вектор развития ИИ сместился в прикладную область: важно оценивать, не просто насоклько хорошо модель генерирует текст, а как она решает конкретные прикладные задачи (код, математика, агентные сценарии). В этих задачах, как правило, корректность можно проверить автоматически и однозначно

## Бенчмарки

Бенчмарк — это стандартизованный набор данных и протокол оценки, позволяющий сравнивать разные модели между собой

Сначала была эпоха отдельных задач: каждый датасет (__SQuAD__ для вопросов-ответов, __SNLI__ для логического следования и так далее) мерил одну узкую способность. Затем пришла эпоха агрегации: бенчмарки __GLUE__ и его усложнённый наследник __SuperGLUE__ собрали россыпь задач на понимание языка в единый набор с одной сводной цифрой, чтобы измерять «понимание языка вообще». Характерно, что обе оценки были вскоре «решены» — модели превысили человеческий уровень, и это стало повторяющимся сюжетом.

Следующая эпоха — знания и рассуждения. Её определяющим бенчмарком стал __MMLU__: 57 предметов от школьного до профессионального уровня, от анатомии до юриспруденции. Рядом встали бенчмарки здравого смысла (__HellaSwag__, ARC, WinoGrande, PIQA) и правдивости (__TruthfulQA__, проверяющий, повторяет ли модель распространённые человеческие заблуждения). Отдельно развивались математика (__GSM8K__ — школьные задачи, MATH — олимпиадные) и код (HumanEval, MBPP)

Параллельно возникли мега-наборы и идея всесторонней оценки. __BIG-bench__ собрал более двухсот разнообразных задач, придуманных сообществом; из него выделили особо трудное подмножество BIG-bench Hard. Проект __HELM__ сместил акцент с одной цифры на многомерность: модель прогоняют по множеству сценариев и меряют не только точность, но и устойчивость, калибровку, смещения, эффективность. Это была важная смена философии — от «кто набрал больше» к «какова модель по совокупности свойств»

К середине 2020-х область столкнулась с кризисом насыщения. Классические бенчмарки — MMLU, HellaSwag, HumanEval — топовые модели стали проходить тесты с результатом выше 90%, и различия между ними утонули в шуме. Ответом стало новое поколение более трудных оценок:

- MMLU-Pro - усложнённый MMLU с десятью вариантами ответа вместо четырёх и обязательной цепочкой рассуждений (хотя к 2026 году и он подходит к насыщению)
- GPQA-Diamond - вопросы уровня PhD по биологии, физике и химии, специально составленные так, чтобы их нельзя было нагуглить; неспециалисты набирают около трети даже с доступом в интернет
- Humanity's Last Exam — около трёх тысяч вопросов экспертного уровня от специалистов разных областей, задуманные так, чтобы оставаться трудными несколько лет
- ARC-AGI и ARC-AGI-2 — задачи на абстракцию и обобщение, нацеленные на «текучий интеллект», а не на эрудицию; первая версия была фактически взята reasoning-моделями к концу 2024 года, что и потребовало второй
- Олимпиадная математика (__AIME__ и подобные) и проекты вроде FrontierMath для самого верхнего уровня

Отдельной и быстро растущей ветвью стали агентные бенчмарки, проверяющие не текст, а действие: SWE-bench и SWE-bench Verified (модель должна решить реальную задачу из репозитория на GitHub так, чтобы прошли тесты), а также GAIA, WebArena, AgentBench и tau-bench, оценивающие работу с инструментами, навигацию и многошаговые сценарии. По мере того как модели обретают агентность, оценка тоже трансформируется от "что модель говорит" к "что модель делает"

## Типы бенчмаркинга

Полезно держать в голове, что бенчмарки различаются сразу по нескольким независимым осям. Понимание этих осей помогает и при чтении новых статей, и на собеседовании, где часто просят систематизировать, а не перечислить.

По измеряемой способности
- Знания и эрудиция;
- рассуждения;
- здравый смысл;
- математика;
- код;
- правдивость и безопасность;
- многоязычность;
- мультимодальность;
- работа с инструментами и агентность;
- длинный контекст

По протоколу предъявления<br>Здесь важен исторический сдвиг. В эпоху GLUE модель дообучали под каждую задачу и мерили дообученную версию. С появлением больших моделей перешли к оценке без дообучения, через формулировку запроса: zero-shot (без примеров), few-shot (несколько примеров прямо в контексте) и с явной просьбой рассуждать пошагово (chain-of-thought). Один и тот же бенчмарк может давать очень разные числа в зависимости от протокола, поэтому сравнивать модели корректно только в одинаковых условиях.

По способу выставления оценки
- автоматическая проверка по совпадению или тестам (дёшево и воспроизводимо, но применимо не везде)
- оценка моделью-судьёй (масштабируемо, но со своими искажениями)
- человеческая оценка (наиболее достоверна, но дорога и медленна)

По статичности. Классический бенчмарк — это фиксированный тестовый набор. Но фиксированный набор рано или поздно утекает в обучающие данные и теряет ценность. Поэтому возникли живые и состязательные форматы: непрерывно обновляемые лидерборды и подходы вроде Dynabench, где люди специально придумывают примеры, на которых текущие модели ошибаются.

По размерности результата. Одна сводная цифра (удобно для ранжирования, но скрывает компромиссы) против холистической многомерной оценки в духе HELM (точность, устойчивость, смещения, эффективность по отдельности).

Отдельно стоит выделить оценку по человеческим предпочтениям в формате арены, потому что это качественно иной подход. Вместо фиксированных задач с известными ответами он измеряет, какой ответ людям субъективно нравится больше. Самый известный пример — __LMArena__ (ранее известная как LMSYS Chatbot Arena): пользователю показывают ответы двух анонимных моделей на его собственный запрос, он выбирает лучший, а из миллионов таких попарных сравнений строится рейтинг по схеме [Elo](https://en.wikipedia.org/wiki/Elo_rating_system) или модели Брэдли–Терри. 

Сила арены в том, что она улавливает «ощущение полезности», которое не видят формальные бенчмарки. Слабость — в том же: люди также не лишины предвзятости: они склонны голосовать за более длинные, уверенные и красиво оформленные ответы, поэтому стиль может побеждать точность, и арена в этом случае измеряет "красоту", а не содержание

## Needle in a Haystack
С развитием моделей произошел взрывной рост максимального окна контекста: 128 тысяч, 200 тысяч, а затем и миллионы токенов. Но возможность загружать длинный текст - это одно, а насколько эффективно модель использует этот контекст - другое

Идею теста "Needle in a Haystack" предложил Грег Камрадт в 2023 году: в длинный текст вставляют одно несвязанное с темой предложение (например, утверждение, что лучшее занятие в Сан-Франциско — съесть сэндвич в парке Долорес в солнечный день) и просят модель ответить на вопрос, связанный с этим утверждением (например, "Какое лучшее занятие в Сан-Франциско?"). Эксперимент многократно повторяют, меняя локацию и общую длину контекста. Затем статистику правильных ответов рисуют тепловой картой: по одной оси — длина, по другой — глубина, цвет ячейки показывает, правльно ли ответила модель

<img src="img/needle_in_a_haystack.png" width=300>

Обнаружили что:
- способность к извлечению деградирует с ростом длины (что довольно очевидно)
- почти всегда возникает эффект __Lost in the middle__: информацию в начале и в конце контекста модели находят почти безошибочно, а вот ближе к середине провал в качестве [(Liu et al, 2023)](https://arxiv.org/abs/2307.03172)

<img src="img/lost_in_the_middle.png" width=300>

Отчасти эффект может объясняться "осторожностью" модели. Так было, например, с [ранними тестами](https://www.anthropic.com/news/claude-2-1-prompting) Claude 2.1, который  набрал низкую точность из-за того, что был обучен не отвечать на основании информации, которую считает недостаточно достоверной. Переформулировка запроса заметно меняла результа

В дальнейшем тест эволюционировал в сторону усложнения. Появились варианты с несколькими фактами (__multi-needle__), требующие интегрировать разрозненную информацию. Возник бенчмарк __RULER__ с набором синтетических длинноконтекстных задач и увидели, что большинство моделей, проходивших оригниальный тест, валятся на нем. 

В том же направлении работают __NeedleBench__, __BABILong__ (рассуждение в длинном контексте), __NoLiMa__ (поиск без буквального совпадения слов) и LongBench

Существует множество методик, как бороьтся с неравномерностью внимания. Некоторые из них:
- __соритровка контекста__<br>расположение более релевантных документов ближе к началу/концу<br>повторное переранжирование отобранных документов более сложной полносвязной моделью
- __калибровка Attention__<br>добавляем числовую поправку, чтобы внимание больше веса давало токенам из середины контекста [(Hsieh et al, 2024)](https://arxiv.org/abs/2406.16008)<br>документы, которым модель часто дает больше внимания, перемещаем ближе к краю [(Peysakhovich et al, 2023)](https://arxiv.org/abs/2310.01427)<br>испольщование RoPE эмбедингов уменьшает деградацию на супер длинных контекстах [(Su et al, 2021)](https://arxiv.org/abs/2104.09864)
- __сжатие контекста__<br>выкидывание малозначимых токенов по перплексии, а-ля TF-IDF [(Jiang et al, 2023)](https://arxiv.org/abs/2310.05736)<br>замена кусков текста на их компактную суммаризацию
- __грамотное обучение__<br>расположение факта в разных локациях [(An et al, 2024)](https://arxiv.org/abs/2404.16811)<br>примеры с инструкциями на реально длинных контекстах
- __грамотный prompt engineering__<br>инструкцию дублируют в начале и конце<br>решают задачу по частям, потом агрегируют (в стиле map-reduce)
- __ориентир на точность__<br>на шаге Retrieval ставят выше порог релевантности - контекст получается меньше [(Lui et al, 2023)](https://arxiv.org/pdf/2407.01100)<br>информацию достают итеративно

## Проблемы и ограничения метрик

Наивная вера в цифры лидербордов — частая ошибка. Ниже список проблем, которые могут возникать у бенчмарков

__Утечки данных__<br>Data contamination - главная беда бенчмарков. Тестовые наборы публичны и со временем попадают в обучающие данные следующих моделей. Тогда высокий балл отражает не способность рассуждать, а запоминание ответов. Именно поэтому так ценятся свежие, приватные и состязательные оценки

__Насыщение__<br>У каждого бенчмарка ограниченный срок жизни: как только модели упираются в его потолок, он перестаёт различать сильнейших, и нужен новый, более трудный. Мы видели это на всей цепочке от GLUE до MMLU-Pro.

__Закон Гудхарта__<br>Когда бенчмарк становится целью оптимизации, под него начинают подгонять обучение, и высокий балл может расходиться с реальной полезностью. Модель учат «сдавать экзамен», а не быть умной

__Валидность__<br>Не всегда очевидно, что бенчмарк измеряет именно ту способность, которую заявляет; формат с выбором из вариантов, например, оставляет лазейки, которые модель может эксплуатировать, не понимая сути

__Разрыв с реальностью__<br>Высокий балл на академическом бенчмарке не гарантирует пользы в реальном продукте; корреляция между лидербордами и фактическим качеством работы бывает слабой, и нередко модели с более скромными общими баллами оказываются точнее на конкретных прикладных задачах.

__Воспроизводимость__<br>Результаты чувствительны к формулировке запроса, числу примеров, версии оценочного инструментария и способу нормализации ответа, поэтому числа из разных источников не всегда сравнимы напрямую.
